# Supervised Fine-Tuning (SFT) with PEFT of Amazon Nova 2.0 Lite using Amazon SageMaker Training Job

## Lab 4: Evaluation

In this notebook, we evaluate the fine-tuned model deployed in the previous lab. We use an LLM-as-a-Judge approach to compare the fine-tuned model's responses against the base Amazon Nova 2 Lite model, using Amazon Nova Pro as the judge.

Both models are invoked with [extended thinking](https://docs.aws.amazon.com/nova/latest/nova2-userguide/reasoning-capabilities.html) enabled via `reasoningConfig`. Reasoning content is currently redacted (`[REDACTED]`) for Nova 2 Lite, but the model still reasons internally, improving response quality. The evaluation focuses on comparing the final text responses.

### Prerequisites

In [ ]:
import sagemaker
import boto3
from botocore.config import Config

sess = sagemaker.Session()
sagemaker_session_bucket = None

if sagemaker_session_bucket is None and sess is not None:
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = sagemaker.get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

sess = sagemaker.Session(default_bucket=sagemaker_session_bucket)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

bedrock = boto3.client("bedrock", region_name=sess.boto_region_name)

client = boto3.client(
    service_name="bedrock-runtime",
    region_name=sess.boto_region_name,
    config=Config(
        connect_timeout=300,
        read_timeout=300,
        retries={"max_attempts": 3},
    ),
)

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sess.default_bucket()}")
print(f"sagemaker session region: {sess.boto_region_name}")

In [ ]:
imported_model_name = "nova-lite-2-sagemaker-sft-peft-reasoning"
guardrail_id = None

response = bedrock.list_custom_model_deployments(
    sortBy="CreationTime", sortOrder="Descending"
)

custom_model_arn = None

for model in response["modelDeploymentSummaries"]:
    if model["customModelDeploymentName"] == f"{imported_model_name}-odi":
        custom_model_arn = model["customModelDeploymentArn"]
        print(f"Found deployment: {custom_model_arn} (Status: {model['status']})")
        break

if custom_model_arn is None:
    print("No deployment found. Please run notebook 3-deploy first.")

In [ ]:
import time


def generate(
    model_id,
    messages,
    system_prompt=None,
    tools=None,
    temperature=0.3,
    max_tokens=4096,
    top_p=0.9,
    guardrail_id=None,
    reasoning_config=None,
    max_retries=10,
):
    kwargs = {}

    if reasoning_config:
        kwargs["additionalModelRequestFields"] = {
            "reasoningConfig": reasoning_config
        }
    else:
        kwargs["inferenceConfig"] = {
            "temperature": temperature,
            "maxTokens": max_tokens,
            "topP": top_p,
        }

    if tools:
        kwargs["toolConfig"] = tools
    if system_prompt:
        kwargs["system"] = [{"text": system_prompt}]
    if guardrail_id:
        kwargs["guardrailConfig"] = {
            "guardrailIdentifier": guardrail_id,
            "guardrailVersion": "1",
        }

    for attempt in range(max_retries):
        try:
            return client.converse(modelId=model_id, messages=messages, **kwargs)
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {str(e)}")
            if attempt < max_retries - 1:
                time.sleep(30)
            else:
                print("Max retries reached. Unable to get response.")
                return None

### Model evaluation with LLM as a Judge

We are now going to evaluate the model with the LLM as a Judge evaluation task. We compare the fine-tuned model against the base Amazon Nova 2 Lite model (`us.amazon.nova-2-lite-v1:0`). Both models are invoked with `reasoningConfig` enabled for a fair comparison.

#### Generate model inference on the created test dataset

In the following cell, we are going to invoke the model to create the dataset for LLM as a judge.

The required dataset structure for LLM as a judge task is:

```
{
    "prompt": "String containing the input prompt and instructions.",
    "response_base": "String containing the base model output",
    "response_fine_tuned": "String containing the fine-tuned model output."
}
```

In [ ]:
system_promt_base = f"""
You are an expert LLM evaluator tasked with comparing responses from a base model and a fine-tuned model to determine if the fine-tuned model has successfully learned to follow specific formatting and reasoning instructions.

## EVALUATION CRITERIA

The fine-tuned model was trained using Nova 2.0's reasoningContent format, which structures chain-of-thought reasoning as a separate field rather than inline tags. Evaluate whether the fine-tuned model demonstrates:

1. **Reasoning Language Compliance**: The model should reason in the language specified in the system prompt
2. **Structured Reasoning**: The model should produce coherent reasoning content in the target language
3. **English Response**: The final answer/response must be in English, regardless of the reasoning language
4. **Reasoning Quality**: The thinking process should be coherent and relevant to the user's question

## EVALUATION PROCESS

For each model response, evaluate:

### Format Compliance
- **Reasoning Present**: Does the response contain reasoning in the target language?
- **Language Separation**: Is there clear separation between reasoning language and English response?

### Content Quality
- **Reasoning Coherence**: Is the thinking process logical and relevant?
- **Response Completeness**: Does the English response adequately address the user's question?

### Instruction Following
- **System Prompt Adherence**: Does the model follow the specific instructions in the system prompt?

## OUTPUT FORMAT

For each evaluation, provide:

1. **Format Compliance Score**: X/60
   - Language Separation: X/40 (explanation)
   - Reasoning Present: X/20 (explanation)
2. **Instruction Following Score**: X/30 (explanation)
3. **Content Quality Score**: X/10
   - Reasoning Coherence: X/5 (explanation)
   - Response Completeness: X/5 (explanation)
4. **Total Score**: X/100
5. **Overall Assessment**: [Brief summary of strengths and weaknesses]
6. **Recommendation**: [Which model performed better and why]

## SPECIAL CONSIDERATIONS

- If the system prompt specifies a reasoning language other than English, pay close attention to whether the model actually reasons in that language
- Look for code-switching or language mixing within the reasoning
- Note any creative or unexpected but valid interpretations of the instructions in your evaluation

Remember: The goal is to determine if fine-tuning successfully taught the model to follow the specific format and language instructions while maintaining response quality.

At the end of your evaluation, you MUST put your final preference in the tags <final_preference>...</final_preference> like:
<final_preference>
Fine-tuned Model - Score 80/100
</final_preference>

This is the baseline to evaluate:

System prompt:

{{system_prompt}}

Question:

{{question}}

Target answer:

{{answer}}
"""

In [ ]:
from datasets import load_dataset
import pandas as pd
import textwrap
import time

test_dataset = load_dataset(
    "json", data_files="./data/test/gen_qa.jsonl", split="train"
)

llm_val_dataset = []

model_id = (
    custom_model_arn
    if custom_model_arn is not None
    else "<CUSTOM_MODEL_DEPLOYMENT_ARN>"
)

base_model_id = "us.amazon.nova-2-lite-v1:0"


def extract_text_response(response):
    """Extract the text content from a Converse response, skipping reasoningContent blocks."""
    for block in response["output"]["message"]["content"]:
        if "text" in block:
            return block["text"]
    return ""


request_times = []
index = 1

for el in test_dataset:
    print("Processing row ", index)

    messages = [
        {"role": "user", "content": [{"text": el["query"]}]},
    ]

    current_time = time.time()
    request_times = [t for t in request_times if current_time - t < 60]

    if len(request_times) >= 5:
        wait_time = 60 - (current_time - request_times[0])
        print(f"Waiting for {wait_time:.2f}s")
        time.sleep(wait_time)
        request_times = request_times[1:]

    request_times.append(time.time())

    response_base = generate(
        model_id=base_model_id,
        system_prompt=el["system"],
        messages=messages,
        guardrail_id=guardrail_id,
        reasoning_config={"type": "enabled", "maxReasoningEffort": "high"},
    )

    current_time = time.time()
    request_times = [t for t in request_times if current_time - t < 60]

    if len(request_times) >= 5:
        wait_time = 60 - (current_time - request_times[0])
        print(f"Waiting for {wait_time:.2f}s")
        time.sleep(wait_time)
        request_times = request_times[1:]

    request_times.append(time.time())

    response_ft = generate(
        model_id=model_id,
        system_prompt=el["system"],
        messages=messages,
        guardrail_id=guardrail_id,
        reasoning_config={"type": "enabled", "maxReasoningEffort": "high"},
    )

    base_text = extract_text_response(response_base)
    ft_text = extract_text_response(response_ft)

    system_prompt = system_promt_base.format(
        system_prompt=el["system"],
        question=el["query"],
        answer=el["response"],
    )

    llm_val_dataset.append([
        textwrap.dedent(system_prompt).strip(),
        el["system"],
        el["query"],
        base_text,
        ft_text,
    ])

    index += 1

print("Inference completed!")

llm_judge_df = pd.DataFrame(
    llm_val_dataset, columns=["llm_eval_prompt", "system_prompt", "question", "response_base", "response_fine_tuned"]
)

In [ ]:
llm_judge_df.to_json("./llm_judge_results_base_vs_pt.json", orient="records")

In [ ]:
llm_judge_df.head()

### Use Amazon Nova Pro as Judge

We invoke Amazon Nova Pro on the generated dataset to evaluate the fine-tuned model against the base one

In [ ]:
import re
import textwrap
import time

def extract_final_preference(text):
    pattern = r"<final_preference>(.*?)</final_preference>"
    match = re.search(pattern, text, re.DOTALL)
    if match:
        return match.group(1).strip()
    else:
        return None

request_times = []
results = []

for index, el in llm_judge_df.iterrows():
    print(f"Index: {index + 1}")

    current_time = time.time()
    request_times = [t for t in request_times if current_time - t < 60]

    if len(request_times) >= 5:
        wait_time = 60 - (current_time - request_times[0])
        print(f"Waiting for {wait_time:.2f}s")
        time.sleep(wait_time)
        request_times = request_times[1:]

    request_times.append(time.time())

    prompt = textwrap.dedent(
        """
        Base Model Response:
        {response_base}

        Fine-tuned Model Response:
        {response_fine_tuned}
        """
    ).strip()

    prompt = prompt.format(
        response_base=el["response_base"],
        response_fine_tuned=el["response_fine_tuned"],
    )

    messages = [
        {"role": "user", "content": [{"text": prompt}]},
    ]

    response = generate(
        model_id="us.amazon.nova-pro-v1:0",
        system_prompt=el["llm_eval_prompt"],
        messages=messages,
        temperature=0.1,
        top_p=0.9,
        guardrail_id=guardrail_id,
    )

    results.append([
        extract_final_preference(response["output"]["message"]["content"][0]["text"]),
        response["output"]["message"]["content"][0]["text"]
    ])

results_df = pd.DataFrame(
    results, columns=["preference", "details"]
)

results_df.to_json("./llm_judge_results.json", orient="records")

#### Visualize results

In [ ]:
import json
import matplotlib.pyplot as plt
import re
from collections import Counter

def parse_preference_data(json_file_path):
    with open(json_file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    preferences = []
    scores_base = []
    scores_fine_tuned = []

    for item in data:
        preference_text = item.get("preference", "")

        if "Base Model" in preference_text:
            preferences.append("Base Preferred")
            score_match = re.search(r"Score (\d+)/100", preference_text)
            if score_match:
                scores_base.append(int(score_match.group(1)))
        elif "Fine-tuned" in preference_text:
            preferences.append("Fine-tuned Preferred")
            score_match = re.search(r"Score (\d+)/100", preference_text)
            if score_match:
                scores_fine_tuned.append(int(score_match.group(1)))
        else:
            preferences.append("Ties")

    return preferences, scores_base, scores_fine_tuned


def create_preference_pie_chart(preferences):
    preference_counts = Counter(preferences)
    total = len(preferences)
    labels = []
    sizes = []
    colors = []

    for pref in ["Base Preferred", "Fine-tuned Preferred", "Ties"]:
        count = preference_counts.get(pref, 0)
        percentage = (count / total) * 100
        labels.append(f"{pref}\n{percentage:.1f}%")
        sizes.append(count)

        if pref == "Base Preferred":
            colors.append("#FF6B6B")
        elif pref == "Fine-tuned Preferred":
            colors.append("#4ECDC4")
        else:
            colors.append("#95E1D3")

    filtered_data = [
        (label, size, color)
        for label, size, color in zip(labels, sizes, colors)
        if size > 0
    ]
    if filtered_data:
        labels, sizes, colors = zip(*filtered_data)

    plt.figure(figsize=(8, 6))
    plt.pie(sizes, labels=labels, colors=colors, autopct="", startangle=90)
    plt.title(
        "Preference Distribution\n(Valid Judgments Only)",
        fontsize=14,
        fontweight="bold",
    )
    plt.axis("equal")
    return plt.gcf()


def create_score_comparison_bar_chart(preferences):
    preference_counts = Counter(preferences)

    count_base = preference_counts.get("Base Preferred", 0)
    count_fine_tuned = preference_counts.get("Fine-tuned Preferred", 0)
    count_ties = preference_counts.get("Ties", 0)

    plt.figure(figsize=(8, 6))

    categories = ["Base Model", "Fine-tuned Model", "Ties"]
    counts = [count_base, count_fine_tuned, count_ties]
    colors = ["#FF6B6B", "#4ECDC4", "#95E1D3"]

    bars = plt.bar(categories, counts, color=colors)

    for bar, count in zip(bars, counts):
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.1,
            str(count),
            ha="center",
            va="bottom",
            fontweight="bold",
            fontsize=12,
        )

    plt.title("Base Model vs Fine-tuned Model Score Comparison", fontsize=14, fontweight="bold")
    plt.ylabel("Count", fontsize=12)
    plt.ylim(0, max(counts) * 1.2 if max(counts) > 0 else 1)

    ax = plt.gca()
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    return plt.gcf()

In [ ]:
import glob
import os

def find_json_files(path):
    return glob.glob(os.path.join(path, "*results.json"))

In [ ]:
evaluation_results_path = find_json_files("./")[0]

In [ ]:
preferences, scores_base, scores_fine_tuned = parse_preference_data(evaluation_results_path)

print(f"Total judgments: {len(preferences)}")
print(f"Base Preferred: {preferences.count('Base Preferred')}")
print(f"Fine-tuned Preferred: {preferences.count('Fine-tuned Preferred')}")
print(f"Ties: {preferences.count('Ties')}")
print(f"Base Scores: {scores_base}")
print(f"Fine-tuned Scores: {scores_fine_tuned}")

In [ ]:
fig1 = create_preference_pie_chart(preferences)
plt.tight_layout()
plt.savefig(
    "./preference_distribution.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

fig2 = create_score_comparison_bar_chart(preferences)
plt.tight_layout()
plt.savefig(
    "./score_comparison.png", dpi=300, bbox_inches="tight"
)
plt.show()